# Federated Learning with Flower (FedAvg): Evidence and Simulation

**Purpose.** Present committed Flower run evidence (parsed from logs) and, when dependencies permit, run a light local federated simulation/reference implementation. We compare centralized vs federated vs local-only behavior and discuss non-IID effects.

## Learning objectives
- Parse and interpret existing FL evidence logs.
- Explain FedAvg aggregation: $w_{t+1}=\sum_k \frac{n_k}{\sum_j n_j} w_{t+1}^k$.
- Compare centralized, FedAvg, and local-only training (light simulation or reference).
- Visualize convergence, per-client heterogeneity, and communication cost.

## Background and math

### Federated Averaging (FedAvg)
Each selected client $k$ performs local updates on $n_k$ samples to obtain $w_{t+1}^k$; server aggregates:
$$
w_{t+1} = \sum_{k\in S_t} \frac{n_k}{n_{S_t}} w_{t+1}^k, \qquad n_{S_t}=\sum_{k\in S_t} n_k.
$$
*Meaning here:* data stays local; only model updates/weights are communicated.

### Non-IID and FedBN
Country/client distributions differ (non-IID), which can slow or bias convergence. FedBN keeps BatchNorm statistics/parameters local while aggregating others (conceptual note here; implementation depends on strategy/model).
*Meaning here:* heterogeneity is expected in this 4-country setup.

## Assumptions and limitations
- Committed logs are evidence of past runs in specific environments; they are labeled as such.
- Optional live Flower/Torch simulation depends on installed packages and runtime; a NumPy logistic-regression FedAvg reference is provided as a clearly labeled fallback.
- Light mode uses few rounds/small data for speed.


In [ ]:
import sys
from pathlib import Path
_here = Path.cwd().resolve()
for _p in [_here, _here / "notebooks", _here.parent]:
    if (_p / "_nb_common.py").exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
        break
import _nb_common as nb
nb.style()
rng = nb.set_seed()
nb.env_report()


In [ ]:
from dataclasses import dataclass
@dataclass(frozen=True)
class Cfg:
    seed: int = nb.SEED
    rounds: int = 3 if nb.MODE == "light" else 10
    local_epochs: int = 1
    n_rows: int = 1200 if nb.MODE == "light" else 4000
cfg = Cfg(); cfg


In [ ]:
import re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

log1 = nb.ROOT / 'docs/evidence/layer1_flwr_run_2rounds.log'
log2 = nb.ROOT / 'docs/evidence/layer2_flwr_run_3rounds.log'
for p in [log1, log2]:
    print(p.name, 'exists=', p.exists())


## Parse committed evidence logs

In [ ]:
def parse_flwr_log(path: Path):
    rows = []
    if not path.exists():
        return pd.DataFrame(rows), [f'missing: {path}']
    text = path.read_text(errors='ignore')
    notes = []
    # round markers and loss lines
    for line in text.splitlines():
        m = re.search(r"\[ROUND\s+(\d+)\]", line)
        if m:
            rows.append({'event':'round', 'round': int(m.group(1)), 'raw': line})
        m2 = re.search(r"round\s+(\d+):\s*([0-9.eE+-]+)", line)
        if m2 and 'History' not in line:
            rows.append({'event':'metric', 'round': int(m2.group(1)), 'metric': 'loss', 'value': float(m2.group(2)), 'raw': line})
        if 'History (loss, distributed)' in line:
            rows.append({'event':'history_header', 'raw': line})
    # parse summary history explicitly
    for m in re.finditer(r"round\s+(\d+):\s*([0-9.eE+-]+)", text):
        # keep unique round/value from summary if not already captured as metric
        pass
    df = pd.DataFrame(rows)
    if df.empty:
        notes.append('no parseable round/loss patterns found')
    return df, notes

df1, n1 = parse_flwr_log(log1)
df2, n2 = parse_flwr_log(log2)
print('log1 notes:', n1); print('log2 notes:', n2)
df1.head(10), df2.head(10)


In [ ]:
# extract distributed loss history if present
def extract_loss_history(path: Path):
    if not path.exists(): return pd.DataFrame()
    text = path.read_text(errors='ignore')
    # capture lines after History (loss, distributed)
    losses = []
    in_hist = False
    for line in text.splitlines():
        if 'History (loss, distributed)' in line:
            in_hist = True; continue
        if in_hist:
            m = re.search(r'round\s+(\d+):\s*([0-9.eE+-]+)', line)
            if m:
                losses.append({'round': int(m.group(1)), 'loss': float(m.group(2))})
            elif line.strip().startswith('INFO') and 'round' not in line:
                # end of block heuristic
                if losses: break
    return pd.DataFrame(losses).drop_duplicates()

h1 = extract_loss_history(log1); h1['source']='layer1_2rounds'
h2 = extract_loss_history(log2); h2['source']='layer2_3rounds'
hist = pd.concat([h1,h2], ignore_index=True) if not h1.empty or not h2.empty else pd.DataFrame()
hist


**How to read this / what it shows.** Parsed distributed loss by round from committed Flower evidence logs. Parsing is regex-based; unparsed details are left in raw logs and not guessed.

## Light simulation / reference implementation (labeled)

In [ ]:
# Prepare data for a simple binary task (is_high_risk)
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, accuracy_score

clients = nb.load_clients(split='train', n_max=cfg.n_rows)
target = 'is_high_risk'
feat_cols = [c for c in next(iter(clients.values())).select_dtypes(include=[np.number]).columns if c != target]
client_data = {}
for cid, df in clients.items():
    d = df[feat_cols + [target]].dropna()
    X = d[feat_cols].values.astype(float); y = d[target].astype(int).values
    # split train/val
    n = len(X); cut = max(1, int(0.8*n))
    client_data[cid] = {'Xtr':X[:cut], 'ytr':y[:cut], 'Xva':X[cut:], 'yva':y[cut:], 'n':cut}
sizes = pd.DataFrame([{ 'client':k, 'country':nb.CLIENT_LABELS.get(k,k), 'n_train':v['n'], 'pos_rate': float(v['ytr'].mean()) if len(v['ytr']) else np.nan} for k,v in client_data.items()])
sizes


In [ ]:
# NumPy logistic-regression FedAvg reference (not Flower) - clearly labeled
def sigmoid(z):
    z = np.clip(z, -30, 30); return 1/(1+np.exp(-z))

def local_sgd(w, b, X, y, lr=0.1, epochs=1, batch=64, seed=0):
    rngl = np.random.default_rng(seed)
    n = len(X)
    for _ in range(epochs):
        idx = rngl.permutation(n)
        for i in range(0,n,batch):
            j = idx[i:i+batch]
            xb,yb = X[j], y[j]
            p = sigmoid(xb@w + b)
            grad_w = xb.T@(p-yb)/len(yb)
            grad_b = np.mean(p-yb)
            w = w - lr*grad_w
            b = b - lr*grad_b
    return w,b

def fedavg(weights, biases, ns):
    ns = np.array(ns, dtype=float); tot = ns.sum()
    w = sum(wi*(ni/tot) for wi,ni in zip(weights,ns))
    b = sum(bi*(ni/tot) for bi,ni in zip(biases,ns))
    return w,b

# preprocess globally for comparability in this reference demo
imp = SimpleImputer(strategy='median'); sc = StandardScaler()
Xall = np.vstack([v['Xtr'] for v in client_data.values()])
imp.fit(Xall); sc.fit(imp.transform(Xall))
for cid in client_data:
    for part in ['Xtr','Xva']:
        client_data[cid][part] = sc.transform(imp.transform(client_data[cid][part]))

# centralized
Xc = np.vstack([v['Xtr'] for v in client_data.values()]); yc = np.concatenate([v['ytr'] for v in client_data.values()])
Xv = np.vstack([v['Xva'] for v in client_data.values()]); yv = np.concatenate([v['yva'] for v in client_data.values()])
w = np.zeros(Xc.shape[1]); b=0.0
cent_hist=[]
for r in range(cfg.rounds):
    w,b = local_sgd(w,b,Xc,yc,lr=0.1,epochs=1,seed=nb.SEED+r)
    p = sigmoid(Xv@w+b); cent_hist.append({'round':r+1,'loss': log_loss(yv,p, labels=[0,1]), 'acc': accuracy_score(yv,(p>=0.5).astype(int)), 'method':'centralized'})

# FedAvg reference
wg = np.zeros(Xc.shape[1]); bg=0.0
fed_hist=[]; per_client_round=[]
for r in range(cfg.rounds):
    ws,bs,ns=[],[],[]
    for i,(cid,v) in enumerate(client_data.items()):
        wl,bl = local_sgd(wg.copy(), float(bg), v['Xtr'], v['ytr'], lr=0.1, epochs=1, seed=nb.SEED+100*r+i)
        ws.append(wl); bs.append(bl); ns.append(v['n'])
    wg,bg = fedavg(ws,bs,ns)
    p = sigmoid(Xv@wg+bg); fed_hist.append({'round':r+1,'loss': log_loss(yv,p, labels=[0,1]), 'acc': accuracy_score(yv,(p>=0.5).astype(int)), 'method':'fedavg_reference'})
    for cid,v in client_data.items():
        pva = sigmoid(v['Xva']@wg+bg)
        per_client_round.append({'round':r+1,'client':cid,'country':nb.CLIENT_LABELS.get(cid,cid),'acc': accuracy_score(v['yva'], (pva>=0.5).astype(int)) if len(v['yva']) else np.nan})

# local-only
loc_hist=[]
for cid,v in client_data.items():
    wl=np.zeros(v['Xtr'].shape[1]); bl=0.0
    for r in range(cfg.rounds):
        wl,bl = local_sgd(wl,bl,v['Xtr'],v['ytr'],lr=0.1,epochs=1,seed=nb.SEED+hash(cid)%1000+r)
    p = sigmoid(v['Xva']@wl+bl)
    loc_hist.append({'client':cid,'country':nb.CLIENT_LABELS.get(cid,cid),'acc': accuracy_score(v['yva'],(p>=0.5).astype(int)) if len(v['yva']) else np.nan, 'method':'local_only'})

sim_hist = pd.DataFrame(cent_hist + fed_hist)
per_client_round_df = pd.DataFrame(per_client_round)
local_df = pd.DataFrame(loc_hist)
sim_hist.tail(), local_df


**Note:** The simulation above is a NumPy logistic-regression FedAvg **reference implementation**, not a Flower runtime execution. It is labeled to avoid confusion with committed Flower logs.

## Figures

In [ ]:
# Figure 1: committed log loss curves
fig, ax = plt.subplots(figsize=(7,4))
if not hist.empty:
    for src, g in hist.groupby('source'):
        ax.plot(g['round'], g['loss'], marker='o', label=f'committed log: {src}')
else:
    ax.text(0.5,0.5,'No parseable loss history found in committed logs', ha='center')
ax.set_title('Flower evidence: distributed loss vs round (parsed)'); ax.set_xlabel('Round'); ax.set_ylabel('Loss'); ax.legend()
fig.tight_layout(); nb.save_fig(fig, 'flower_evidence_loss'); plt.show()


**How to read this / what it shows.** Loss trajectories parsed from committed evidence logs (if available). These reflect the recorded runs, not necessarily this environment.

In [ ]:
# Figure 2: simulation convergence centralized vs fedavg reference
fig, axes = plt.subplots(1,2, figsize=(11,4))
for method, g in sim_hist.groupby('method'):
    axes[0].plot(g['round'], g['loss'], marker='o', label=method)
    axes[1].plot(g['round'], g['acc'], marker='o', label=method)
axes[0].set_title('Global/val loss (reference sim)'); axes[0].set_xlabel('Round'); axes[0].set_ylabel('Log loss'); axes[0].legend()
axes[1].set_title('Global/val accuracy (reference sim)'); axes[1].set_xlabel('Round'); axes[1].set_ylabel('Accuracy'); axes[1].legend()
fig.tight_layout(); nb.save_fig(fig, 'flower_sim_convergence'); plt.show()


**How to read this / what it shows.** Comparison of centralized and FedAvg-reference convergence on the same simple task/split. Differences illustrate aggregation/heterogeneity effects in this controlled setting.

In [ ]:
# Figure 3: per-client accuracy and data distribution
fig, axes = plt.subplots(1,2, figsize=(11,4))
for cid, g in per_client_round_df.groupby('client'):
    axes[0].plot(g['round'], g['acc'], marker='o', label=nb.CLIENT_LABELS.get(cid,cid))
axes[0].set_title('Per-client val accuracy by round (FedAvg ref)'); axes[0].set_xlabel('Round'); axes[0].set_ylabel('Accuracy'); axes[0].legend()
axes[1].bar(sizes['country'], sizes['n_train'], color='#6a4c93')
ax2 = axes[1].twinx(); ax2.plot(sizes['country'], sizes['pos_rate'], 'o--', color='#d1495b')
axes[1].set_title('Client sizes and positive rates (non-IID indicators)'); axes[1].set_ylabel('Train samples'); ax2.set_ylabel('Positive rate')
fig.tight_layout(); nb.save_fig(fig, 'flower_client_heterogeneity'); plt.show()


**How to read this / what it shows.** Left: per-client performance can diverge under heterogeneity. Right: sample sizes and label rates differ by client, indicating non-IID conditions.

In [ ]:
# Figure 4: communication cost estimate
# bytes = params * 4 bytes * 2 (down+up) * clients * rounds (rough FedAvg full-model exchange)
n_params = Xc.shape[1] + 1
comm_rows=[]
for clients_n in [2,4]:
    for rounds_n in [1,2,3,5,10]:
        bytes_est = n_params * 4 * 2 * clients_n * rounds_n
        comm_rows.append({'clients':clients_n,'rounds':rounds_n,'est_bytes':bytes_est,'est_kib':bytes_est/1024})
comm = pd.DataFrame(comm_rows)
fig, ax = plt.subplots(figsize=(7,4))
for c, g in comm.groupby('clients'):
    ax.plot(g['rounds'], g['est_kib'], marker='o', label=f'{c} clients')
ax.set_title('Estimated communication cost (full model, float32, ref)'); ax.set_xlabel('Rounds'); ax.set_ylabel('KiB (estimated)'); ax.legend()
fig.tight_layout(); nb.save_fig(fig, 'flower_comm_cost'); plt.show()
comm.pivot(index='rounds', columns='clients', values='est_kib')


**How to read this / what it shows.** Analytic estimate of parameter exchange volume; real systems include overhead, compression, partial participation, and secure aggregation costs.

## Quantitative summary


In [ ]:
summary = {
    'evidence_log1': str(log1.exists()),
    'evidence_log2': str(log2.exists()),
    'parsed_loss_points': int(len(hist)) if not hist.empty else 0,
    'ref_fedavg_final_acc': float(sim_hist[sim_hist.method=='fedavg_reference']['acc'].iloc[-1]) if not sim_hist.empty else None,
    'ref_centralized_final_acc': float(sim_hist[sim_hist.method=='centralized']['acc'].iloc[-1]) if not sim_hist.empty else None,
}
pd.DataFrame([summary])


## Sensitivity / ablation
- More rounds/local epochs can improve fit but may increase client drift under non-IID data.
- Participation/stratification and personalization (e.g. FedBN, FedProx) affect convergence.
- Communication cost scales with model size, clients, and rounds.

## Limitations and failure cases
- Flower runtime simulation is not executed here unless explicitly available; reference sim is not Flower.
- Log parsing is best-effort; unrecognized formats are reported rather than inferred.
- Simple linear model results do not represent deep multi-task NN performance.

## Conclusions and further experiments
- Committed evidence logs provide historical FL run traces; this notebook parses and visualizes available signals.
- A labeled FedAvg reference illustrates centralized vs federated vs local-only behavior on bundled data.
- Further work: run live Flower simulations in-environment, log per-round per-client metrics, and evaluate FedProx/FedBN strategies.

## References
- B. McMahan et al. "Communication-Efficient Learning of Deep Networks from Decentralized Data." *AISTATS*, 2017.
- T. Li et al. "Federated Optimization in Heterogeneous Networks (FedProx)." *MLSys*, 2020.
- X. Li et al. "FedBN: Federated Learning on Non-IID Features via Local Batch Normalization." *ICLR*, 2021.
